<!--nav--> [🗺 Learning path](README.md) · **3/39** · ◀ [Simple MultiGPU ActualTraining](./Simple_MultiGPU_ActualTraining.ipynb) · [LoRA QLoRA FineTuning](./LoRA_QLoRA_FineTuning.ipynb) ▶

# Multi-GPU Benchmark: 1 GPU vs 2 GPU vs Strategies

Run the same training on different GPU configs and parallelism strategies.
See exactly how much speedup you get.

### What we benchmark

```
Run 1:  1 GPU,  no DeepSpeed        (baseline)
Run 2:  2 GPU,  DDP                  (data parallelism)
Run 3:  2 GPU,  DeepSpeed ZeRO-2     (+ optimizer sharding)
Run 4:  2 GPU,  DeepSpeed ZeRO-3     (+ model sharding)
```

Same model, same data, same hyperparams. Only the parallelism changes.

In [ ]:
!pip install -q transformers datasets peft accelerate deepspeed matplotlib

In [ ]:
import torch, os, json, time

assert torch.cuda.is_available(), "GPU required!"
NUM_GPUS = torch.cuda.device_count()

for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.0f} GB)")
print(f"\nTotal GPUs: {NUM_GPUS}")
assert NUM_GPUS >= 2, "Need 2+ GPUs to benchmark multi-GPU. Use Kaggle 2x T4."

## Write the Training Script

One script for all runs. It reads a config file to know which strategy to use.
Saves timing + memory metrics to a JSON file after each run.

In [ ]:
%%writefile bench_train.py
"""Benchmark training script — reads run_config.json for strategy."""
import torch, os, json, time
os.environ["WANDB_DISABLED"] = "true"

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback,
)
from peft import LoraConfig, get_peft_model

# Read run config
with open("run_config.json") as f:
    cfg = json.load(f)

RUN_NAME = cfg["name"]
MODEL = "gpt2"

# Load model + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16)

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["c_attn"], bias="none", task_type="CAUSAL_LM",
))

# Load data — same 1000 examples every run
dataset = load_dataset("tatsu-lab/alpaca", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))

def tokenize(ex):
    text = f"### Instruction:\n{ex['instruction']}\n### Response:\n{ex['output']}{tokenizer.eos_token}"
    return tokenizer(text, truncation=True, max_length=256, padding=False)

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# Metrics callback
class BenchCallback(TrainerCallback):
    def __init__(self):
        self.step_times = []
        self.start = None
        self.losses = []

    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()
        torch.cuda.reset_peak_memory_stats()

    def on_step_end(self, args, state, control, **kwargs):
        self.step_times.append(time.time())

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.losses.append({"step": state.global_step, "loss": logs["loss"]})

    def on_train_end(self, args, state, control, **kwargs):
        if int(os.environ.get("LOCAL_RANK", 0)) != 0:
            return
        train_time = time.time() - self.start
        num_gpus = int(os.environ.get("WORLD_SIZE", 1))
        total_steps = state.global_step
        total_samples = total_steps * args.per_device_train_batch_size * args.gradient_accumulation_steps * num_gpus

        if len(self.step_times) > 2:
            durs = [self.step_times[i+1] - self.step_times[i] for i in range(len(self.step_times)-1)]
            avg_step = sum(durs) / len(durs)
        else:
            avg_step = train_time / max(total_steps, 1)

        result = {
            "name": RUN_NAME,
            "num_gpus": num_gpus,
            "gpu_name": torch.cuda.get_device_name(0),
            "train_time_sec": round(train_time, 2),
            "total_steps": total_steps,
            "total_samples": total_samples,
            "samples_per_sec": round(total_samples / train_time, 2),
            "avg_step_time_ms": round(avg_step * 1000, 1),
            "gpu_mem_allocated_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "gpu_mem_reserved_gb": round(torch.cuda.max_memory_reserved() / 1e9, 2),
            "gpu_mem_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "final_loss": self.losses[-1]["loss"] if self.losses else None,
            "loss_history": self.losses,
        }
        outfile = f"bench_{RUN_NAME.replace(' ', '_').lower()}.json"
        with open(outfile, "w") as f:
            json.dump(result, f, indent=2)
        print(f"\n{RUN_NAME}: {train_time:.1f}s | {total_samples/train_time:.1f} samples/sec | {torch.cuda.max_memory_allocated()/1e9:.2f} GB peak")

cb = BenchCallback()

# Same training config for every run — only parallelism differs
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=f"./bench_output/{RUN_NAME}",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=5,
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[cb],
)

trainer.train()

## Run 1: Baseline — 1 GPU, No DeepSpeed

Plain single-GPU training. This is what you'd get on Colab free tier.

In [ ]:
# Write accelerate config for 1 GPU, no DeepSpeed
def write_accel_config(num_procs, strategy, zero_stage=None):
    """Write accelerate config for a given strategy."""
    accel_dir = os.path.expanduser("~/.cache/huggingface/accelerate")
    os.makedirs(accel_dir, exist_ok=True)
    path = os.path.join(accel_dir, "default_config.yaml")

    if strategy == "none":
        yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: 'NO'
machine_rank: 0
mixed_precision: bf16
num_machines: 1
num_processes: 1
use_cpu: false
"""
    elif strategy == "ddp":
        yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
machine_rank: 0
mixed_precision: bf16
num_machines: 1
num_processes: {num_procs}
use_cpu: false
"""
    elif strategy == "zero2":
        yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: none
  zero3_init_flag: false
  zero_stage: 2
machine_rank: 0
mixed_precision: bf16
num_machines: 1
num_processes: {num_procs}
use_cpu: false
"""
    elif strategy == "zero3":
        yaml = f"""compute_environment: LOCAL_MACHINE
distributed_type: DEEPSPEED
deepspeed_config:
  gradient_accumulation_steps: auto
  gradient_clipping: auto
  offload_optimizer_device: cpu
  offload_param_device: cpu
  zero3_init_flag: true
  zero3_save_16bit_model: true
  zero_stage: 3
machine_rank: 0
mixed_precision: bf16
num_machines: 1
num_processes: {num_procs}
use_cpu: false
"""
    with open(path, "w") as f:
        f.write(yaml)

print("Config helper ready.")

In [ ]:
# --- Run 1: 1 GPU baseline ---
write_accel_config(1, "none")
with open("run_config.json", "w") as f:
    json.dump({"name": "1 GPU baseline"}, f)

print("Run 1: 1 GPU, no parallelism (baseline)")
!accelerate launch --num_processes=1 bench_train.py

## Run 2: 2 GPU — DDP (Data Parallelism)

Each GPU gets a copy of the model + processes different data batches.
Gradients are averaged across GPUs. Simplest multi-GPU strategy.

In [ ]:
# --- Run 2: 2 GPU DDP ---
write_accel_config(NUM_GPUS, "ddp")
with open("run_config.json", "w") as f:
    json.dump({"name": "2 GPU DDP"}, f)

print(f"Run 2: {NUM_GPUS} GPU, DDP (data parallelism)")
!accelerate launch --num_processes={NUM_GPUS} bench_train.py

## Run 3: 2 GPU — DeepSpeed ZeRO-2

Same as DDP but optimizer states + gradients are sharded across GPUs.
Less memory per GPU → can use bigger batches or bigger models.

In [ ]:
# --- Run 3: 2 GPU ZeRO-2 ---
write_accel_config(NUM_GPUS, "zero2")
with open("run_config.json", "w") as f:
    json.dump({"name": "2 GPU ZeRO-2"}, f)

print(f"Run 3: {NUM_GPUS} GPU, DeepSpeed ZeRO-2 (optimizer sharding)")
!accelerate launch --num_processes={NUM_GPUS} bench_train.py

## Run 4: 2 GPU — DeepSpeed ZeRO-3 (Model Parallelism)

Model weights are also sharded. Each GPU only holds a fraction.
More communication overhead but can fit much bigger models.

In [ ]:
# --- Run 4: 2 GPU ZeRO-3 ---
write_accel_config(NUM_GPUS, "zero3")
with open("run_config.json", "w") as f:
    json.dump({"name": "2 GPU ZeRO-3"}, f)

print(f"Run 4: {NUM_GPUS} GPU, DeepSpeed ZeRO-3 (model parallelism)")
!accelerate launch --num_processes={NUM_GPUS} bench_train.py

---
## Performance Dashboard

In [ ]:
import json, glob
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import HTML, display

# Load all benchmark results
results = []
for f in sorted(glob.glob("bench_*.json")):
    with open(f) as fh:
        results.append(json.load(fh))

names = [r["name"] for r in results]
times = [r["train_time_sec"] for r in results]
throughputs = [r["samples_per_sec"] for r in results]
step_times = [r["avg_step_time_ms"] for r in results]
mem_alloc = [r["gpu_mem_allocated_gb"] for r in results]
baseline_time = times[0]
speedups = [baseline_time / t for t in times]

colors = ["#8b949e", "#58a6ff", "#3fb950", "#d2a8ff"]

# ── Chart 1: Training Time + Speedup ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("Multi-GPU Benchmark Results", color="#e6edf3", fontsize=16, fontweight="bold", y=1.02)

# Training time bars
ax1.set_facecolor("#0d1117")
bars1 = ax1.bar(names, times, color=colors, width=0.6, edgecolor="#0d1117", linewidth=2)
for bar, t, s in zip(bars1, times, speedups):
    label = f"{t:.0f}s"
    if s > 1.01:
        label += f"\n{s:.2f}x"
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(times)*0.02,
             label, ha="center", va="bottom", color="#e6edf3", fontsize=11, fontweight="bold")
ax1.set_ylabel("Training Time (seconds)", color="#8b949e", fontsize=11)
ax1.set_title("Training Time (lower = better)", color="#e6edf3", fontsize=13)
ax1.tick_params(colors="#8b949e")
ax1.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax1.grid(True, axis="y", alpha=0.15, color="#30363d")
for spine in ax1.spines.values():
    spine.set_color("#30363d")

# Throughput bars
ax2.set_facecolor("#0d1117")
bars2 = ax2.bar(names, throughputs, color=colors, width=0.6, edgecolor="#0d1117", linewidth=2)
for bar, tp in zip(bars2, throughputs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(throughputs)*0.02,
             f"{tp:.1f}", ha="center", va="bottom", color="#e6edf3", fontsize=11, fontweight="bold")
ax2.set_ylabel("Samples / Second", color="#8b949e", fontsize=11)
ax2.set_title("Throughput (higher = better)", color="#e6edf3", fontsize=13)
ax2.tick_params(colors="#8b949e")
ax2.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax2.grid(True, axis="y", alpha=0.15, color="#30363d")
for spine in ax2.spines.values():
    spine.set_color("#30363d")

plt.tight_layout()
plt.show()

# ── Chart 2: Memory + Step Time ──
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 4))
fig2.patch.set_facecolor("#0d1117")

# GPU memory
ax3.set_facecolor("#0d1117")
bars3 = ax3.bar(names, mem_alloc, color=colors, width=0.6, edgecolor="#0d1117", linewidth=2)
gpu_total = results[0]["gpu_mem_total_gb"]
ax3.axhline(y=gpu_total, color="#f85149", linestyle="--", alpha=0.5, label=f"GPU limit ({gpu_total} GB)")
for bar, m in zip(bars3, mem_alloc):
    pct = m / gpu_total * 100
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + gpu_total*0.02,
             f"{m:.1f} GB\n({pct:.0f}%)", ha="center", va="bottom", color="#e6edf3", fontsize=10, fontweight="bold")
ax3.set_ylabel("Peak GPU Memory (GB)", color="#8b949e", fontsize=11)
ax3.set_title("GPU Memory per GPU (lower = fits bigger models)", color="#e6edf3", fontsize=13)
ax3.tick_params(colors="#8b949e")
ax3.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax3.set_ylim(0, gpu_total * 1.35)
ax3.legend(facecolor="#161b22", edgecolor="#30363d", labelcolor="#e6edf3", fontsize=9)
ax3.grid(True, axis="y", alpha=0.15, color="#30363d")
for spine in ax3.spines.values():
    spine.set_color("#30363d")

# Average step time
ax4.set_facecolor("#0d1117")
bars4 = ax4.bar(names, step_times, color=colors, width=0.6, edgecolor="#0d1117", linewidth=2)
for bar, st in zip(bars4, step_times):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(step_times)*0.02,
             f"{st:.0f} ms", ha="center", va="bottom", color="#e6edf3", fontsize=11, fontweight="bold")
ax4.set_ylabel("Step Time (ms)", color="#8b949e", fontsize=11)
ax4.set_title("Avg Step Time (lower = faster)", color="#e6edf3", fontsize=13)
ax4.tick_params(colors="#8b949e")
ax4.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax4.grid(True, axis="y", alpha=0.15, color="#30363d")
for spine in ax4.spines.values():
    spine.set_color("#30363d")

plt.tight_layout()
plt.show()

# ── Chart 3: Loss curves overlaid ──
fig3, ax5 = plt.subplots(figsize=(10, 4))
fig3.patch.set_facecolor("#0d1117")
ax5.set_facecolor("#0d1117")

for r, c in zip(results, colors):
    if r["loss_history"]:
        s = [h["step"] for h in r["loss_history"]]
        l = [h["loss"] for h in r["loss_history"]]
        ax5.plot(s, l, color=c, linewidth=2, label=r["name"], alpha=0.85)

ax5.set_xlabel("Step", color="#8b949e", fontsize=11)
ax5.set_ylabel("Loss", color="#8b949e", fontsize=11)
ax5.set_title("Training Loss — All Strategies", color="#e6edf3", fontsize=14, fontweight="bold")
ax5.tick_params(colors="#8b949e")
ax5.grid(True, alpha=0.15, color="#30363d")
ax5.legend(facecolor="#161b22", edgecolor="#30363d", labelcolor="#e6edf3", fontsize=10)
for spine in ax5.spines.values():
    spine.set_color("#30363d")
plt.tight_layout()
plt.show()

In [ ]:
# ── HTML Summary Table ──
rows_html = ""
for i, r in enumerate(results):
    speedup = baseline_time / r["train_time_sec"]
    mem_pct = r["gpu_mem_allocated_gb"] / r["gpu_mem_total_gb"] * 100
    is_best_speed = r["samples_per_sec"] == max(throughputs)
    is_best_mem = r["gpu_mem_allocated_gb"] == min(mem_alloc)

    speed_color = "#3fb950" if is_best_speed else "#c9d1d9"
    mem_color = "#3fb950" if is_best_mem else "#c9d1d9"
    speedup_badge = ""
    if speedup > 1.01:
        speedup_badge = f'<span style="background:#238636; color:white; padding:2px 8px; border-radius:10px; font-size:11px; margin-left:6px;">{speedup:.2f}x</span>'

    rows_html += f"""
    <tr style="border-bottom: 1px solid #21262d;">
      <td style="padding:12px; font-weight:600; color:{colors[i]};">{r['name']}</td>
      <td style="padding:12px; text-align:center;">{r['num_gpus']}</td>
      <td style="padding:12px; text-align:center;">{r['train_time_sec']:.0f}s {speedup_badge}</td>
      <td style="padding:12px; text-align:center; color:{speed_color}; font-weight:600;">{r['samples_per_sec']:.1f}</td>
      <td style="padding:12px; text-align:center;">{r['avg_step_time_ms']:.0f} ms</td>
      <td style="padding:12px; text-align:center; color:{mem_color};">{r['gpu_mem_allocated_gb']:.1f} GB ({mem_pct:.0f}%)</td>
      <td style="padding:12px; text-align:center;">{r['final_loss']:.3f}</td>
    </tr>"""

best_speedup = max(speedups)
best_strat = names[speedups.index(best_speedup)]
mem_saved = (mem_alloc[0] - min(mem_alloc)) / mem_alloc[0] * 100

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 850px; margin: 20px 0;">

  <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 16px;">
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 18px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Best Speedup</div>
      <div style="color: #3fb950; font-size: 30px; font-weight: 700; margin: 6px 0;">{best_speedup:.2f}x</div>
      <div style="color: #8b949e; font-size: 12px;">{best_strat}</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 18px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Peak Throughput</div>
      <div style="color: #58a6ff; font-size: 30px; font-weight: 700; margin: 6px 0;">{max(throughputs):.1f}</div>
      <div style="color: #8b949e; font-size: 12px;">samples/sec</div>
    </div>
    <div style="background: linear-gradient(135deg, #1a2332, #161b22); border: 1px solid #30363d;
                border-radius: 12px; padding: 18px; text-align: center;">
      <div style="color: #8b949e; font-size: 11px; text-transform: uppercase; letter-spacing: 1px;">Memory Saved</div>
      <div style="color: #d2a8ff; font-size: 30px; font-weight: 700; margin: 6px 0;">{mem_saved:.0f}%</div>
      <div style="color: #8b949e; font-size: 12px;">vs baseline (best strategy)</div>
    </div>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px; overflow: hidden;">
    <table style="width:100%; border-collapse: collapse; color: #c9d1d9; font-size: 13px;">
      <thead>
        <tr style="background: #0d1117; border-bottom: 2px solid #30363d;">
          <th style="padding:12px; text-align:left; color:#8b949e;">Strategy</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">GPUs</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">Time</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">Throughput</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">Step Time</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">GPU Memory</th>
          <th style="padding:12px; text-align:center; color:#8b949e;">Final Loss</th>
        </tr>
      </thead>
      <tbody>
        {rows_html}
      </tbody>
    </table>
  </div>

  <div style="background: #161b22; border: 1px solid #30363d; border-radius: 12px;
              padding: 16px; margin-top: 12px; color: #8b949e; font-size: 12px;">
    <strong style="color: #e6edf3;">Key takeaways:</strong><br>
    <span style="color:#58a6ff;">DDP</span> — fastest for small models (minimal overhead)<br>
    <span style="color:#3fb950;">ZeRO-2</span> — same speed + less memory (optimizer offloaded to CPU)<br>
    <span style="color:#d2a8ff;">ZeRO-3</span> — most memory efficient (model sharded) but more communication overhead<br>
    All strategies reach the same loss — parallelism doesn't affect convergence.
  </div>

</div>
"""
display(HTML(html))

---

## What Each Strategy Does

```
1 GPU Baseline:
  GPU 0: [model] [optimizer] [gradients] [data]

DDP (2 GPU):
  GPU 0: [model copy] [optimizer] [gradients] [data batch A]
  GPU 1: [model copy] [optimizer] [gradients] [data batch B]
  → Each GPU processes different data, gradients averaged
  → 2x data throughput, same memory per GPU

ZeRO-2 (2 GPU):
  GPU 0: [model copy] [optimizer HALF] [gradients HALF] [data batch A]
  GPU 1: [model copy] [optimizer HALF] [gradients HALF] [data batch B]
  → Optimizer + gradients sharded → less memory per GPU
  → Can increase batch size or use bigger models

ZeRO-3 (2 GPU):
  GPU 0: [model HALF] [optimizer HALF] [gradients HALF] [data batch A]
  GPU 1: [model HALF] [optimizer HALF] [gradients HALF] [data batch B]
  → Everything sharded → minimum memory per GPU
  → Model params gathered on-the-fly when needed
  → More communication but fits biggest models
```

### When to use what

| Strategy | Best for |
|----------|----------|
| **DDP** | Small models that fit on 1 GPU — fastest, least overhead |
| **ZeRO-2** | Medium models, or when you need bigger batches |
| **ZeRO-3** | Large models that don't fit on a single GPU |

All three give ~2x throughput on 2 GPUs. The difference is memory.